## Setup: Directory Configuration and Library Imports


In [90]:
import pandas as pd
from pathlib import Path

In [91]:
# Configure paths
PREPROCESSED_DIR = Path('../../data/preprocessed')
MERGED_DIR = Path('../../data/merged')

# Create output directory if it doesn't exist
MERGED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Preprocessed data location: {PREPROCESSED_DIR.resolve()}")
print(f"Output location: {MERGED_DIR.resolve()}")
print(f"Output directory created: {MERGED_DIR.exists()}")

Preprocessed data location: C:\Users\Dell\Documents\MSc AAI\FAIDM - WM9QG\Group Assessment\data-mining\data\preprocessed
Output location: C:\Users\Dell\Documents\MSc AAI\FAIDM - WM9QG\Group Assessment\data-mining\data\merged
Output directory created: True


## Step 1: Merging Demographics & Engagement Data

We integrate student demographic information, course registration details, VLE interaction metrics, and course metadata through a series of left joins starting with studentInfo as the base table.


### Loading Preprocessed Tables


In [92]:
# Load student information (demographics)
print("Loading preprocessed tables...")
student_info = pd.read_csv(PREPROCESSED_DIR / 'studentInfo.csv')
print(f"studentInfo: {student_info.shape}")
print(student_info.info())

# Load student registration data
student_reg = pd.read_csv(PREPROCESSED_DIR / 'studentRegistration.csv')
print(f"\nstudentRegistration: {student_reg.shape}")
print(student_reg.info())

Loading preprocessed tables...
studentInfo: (32593, 12)
<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   code_module           32593 non-null  str  
 1   code_presentation     32593 non-null  str  
 2   id_student            32593 non-null  int64
 3   gender                32593 non-null  str  
 4   region                32593 non-null  str  
 5   highest_education     32593 non-null  str  
 6   imd_band              32593 non-null  str  
 7   age_band              32593 non-null  str  
 8   num_of_prev_attempts  32593 non-null  int64
 9   studied_credits       32593 non-null  int64
 10  disability            32593 non-null  str  
 11  final_result          32593 non-null  str  
dtypes: int64(3), str(9)
memory usage: 3.0 MB
None

studentRegistration: (32593, 5)
<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data column

In [93]:
# Load student VLE interaction data (aggregated)
student_vle = pd.read_csv(PREPROCESSED_DIR / 'studentVle.csv')
print(f"studentVle: {student_vle.shape}")
print(student_vle.info())

# Load course metadata
courses = pd.read_csv(PREPROCESSED_DIR / 'courses.csv')
print(f"\ncourses: {courses.shape}")
print(courses.info())

studentVle: (29228, 6)
<class 'pandas.DataFrame'>
RangeIndex: 29228 entries, 0 to 29227
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code_module         29228 non-null  str    
 1   code_presentation   29228 non-null  str    
 2   id_student          29228 non-null  int64  
 3   total_clicks        29228 non-null  int64  
 4   avg_clicks_per_day  29228 non-null  float64
 5   num_sites           29228 non-null  int64  
dtypes: float64(1), int64(3), str(2)
memory usage: 1.3 MB
None

courses: (22, 3)
<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 3 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   code_module                 22 non-null     str  
 1   code_presentation           22 non-null     str  
 2   module_presentation_length  22 non-null     int64
dtypes: int64(1), str(2)
memory usage: 66

### Performing Left Joins

Starting with studentInfo, we perform sequential left joins to add registration data, VLE interactions, and course metadata.


In [94]:
# Initialize master dataframe with studentInfo
master = student_info.copy()
print(f"Initial master dataframe shape: {master.shape}")

# Define join keys
join_keys = ['id_student', 'code_module', 'code_presentation']
print(f"Join keys: {join_keys}")

Initial master dataframe shape: (32593, 12)
Join keys: ['id_student', 'code_module', 'code_presentation']


In [95]:
# Left join with student registration
print("Merging studentRegistration...")
master = master.merge(
    student_reg,
    on=join_keys,
    how='left',
    suffixes=('', '_reg')
)
print(f"After registration merge: {master.shape}")
print(f"Columns added: {[col for col in master.columns if 'reg' in col or 'date_' in col]}")
print(master.info())

Merging studentRegistration...
After registration merge: (32593, 14)
Columns added: ['region', 'date_registration', 'date_unregistration']
<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   code_module           32593 non-null  str    
 1   code_presentation     32593 non-null  str    
 2   id_student            32593 non-null  int64  
 3   gender                32593 non-null  str    
 4   region                32593 non-null  str    
 5   highest_education     32593 non-null  str    
 6   imd_band              32593 non-null  str    
 7   age_band              32593 non-null  str    
 8   num_of_prev_attempts  32593 non-null  int64  
 9   studied_credits       32593 non-null  int64  
 10  disability            32593 non-null  str    
 11  final_result          32593 non-null  str    
 12  date_registration     32548 non-null  float6

In [96]:
# Left join with student VLE data
print("Merging studentVle...")
master = master.merge(
    student_vle,
    on=join_keys,
    how='left',
    suffixes=('', '_vle')
)
print(f"After VLE merge: {master.shape}")
print(f"VLE columns: {[col for col in master.columns if 'click' in col]}")
print(master.isna().sum())

Merging studentVle...
After VLE merge: (32593, 17)
VLE columns: ['total_clicks', 'avg_clicks_per_day']
code_module                 0
code_presentation           0
id_student                  0
gender                      0
region                      0
highest_education           0
imd_band                    0
age_band                    0
num_of_prev_attempts        0
studied_credits             0
disability                  0
final_result                0
date_registration          45
date_unregistration     22521
total_clicks             3365
avg_clicks_per_day       3365
num_sites                3365
dtype: int64


In [97]:
# Left join with courses metadata
print("Merging courses...")
master = master.merge(
    courses[['code_module', 'code_presentation', 'module_presentation_length']],
    on=['code_module', 'code_presentation'],
    how='left'
)
print(f"After courses merge: {master.shape}")
print(f"Length column added: 'module_presentation_length' present = {'module_presentation_length' in master.columns}")
print(master.isna().sum())

Merging courses...
After courses merge: (32593, 18)
Length column added: 'module_presentation_length' present = True
code_module                       0
code_presentation                 0
id_student                        0
gender                            0
region                            0
highest_education                 0
imd_band                          0
age_band                          0
num_of_prev_attempts              0
studied_credits                   0
disability                        0
final_result                      0
date_registration                45
date_unregistration           22521
total_clicks                   3365
avg_clicks_per_day             3365
num_sites                      3365
module_presentation_length        0
dtype: int64


### Validation: Missing Data Analysis


In [98]:
# Check for missing values
print("Missing values by column:")
missing_data = master.isnull().sum()
missing_pct = (master.isnull().sum() / len(master)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_data,
    'Missing_Percentage': missing_pct
}).sort_values('Missing_Count', ascending=False)

print(missing_df[missing_df['Missing_Count'] > 0])
print(f"\nTotal rows with complete data: {master.isnull().sum(axis=1).eq(0).sum()}")

Missing values by column:
                     Missing_Count  Missing_Percentage
date_unregistration          22521           69.097659
total_clicks                  3365           10.324303
num_sites                     3365           10.324303
avg_clicks_per_day            3365           10.324303
date_registration               45            0.138066

Total rows with complete data: 7095


### Ghost Student (with no date_registration)


In [99]:
# Create a dataframe of just the students missing a registration date
ghost_students = master[master['date_registration'].isnull()]

print(f"--- Investigation of {len(ghost_students)} students missing date_registration ---")

# 1. Did any of them withdraw?
withdrawn_ghosts = ghost_students['date_unregistration'].notnull().sum()
print(f"Students who 'un-registered' without a registration date: {withdrawn_ghosts}")

# 2. Did any of them actually use the VLE?
active_ghosts = (ghost_students['total_clicks'] > 0).sum()
print(f"Students who have VLE clicks but no registration date: {active_ghosts}")

# 3. What were their final results?
print("\nFinal Results of these 45 students:")
print(ghost_students['final_result'].value_counts())

# 4. Show a sample of these records
ghost_students[['id_student', 'code_module', 'code_presentation', 'final_result', 'total_clicks']].head()

--- Investigation of 45 students missing date_registration ---
Students who 'un-registered' without a registration date: 39
Students who have VLE clicks but no registration date: 7

Final Results of these 45 students:
final_result
Withdrawn    39
Fail          5
Pass          1
Name: count, dtype: int64


,id_student,code_module,code_presentation,final_result,total_clicks
2344,630346,BBB,2013B,Fail,NaN
2538,57369,BBB,2013J,Withdrawn,NaN
2759,342678,BBB,2013J,Withdrawn,NaN
5356,582496,BBB,2014B,Withdrawn,NaN
5490,607646,BBB,2014B,Withdrawn,NaN


### Handling Missing Data


1. Drop records with missing date_registration as it is non-negligible, might be fake and will hinder further merging and precision of model by feeding incorrect information.


In [100]:
master = master.dropna(subset=['date_registration'])
print(f"Dropped 45 rows. New shape: {master.shape}")

Dropped 45 rows. New shape: (32548, 18)


2. Fill missing VLE interaction metrics (total_clicks, avg_clicks_per_day, num_sites) with 0, as these represent students with no VLE activity.


In [101]:
# Identify VLE columns to fill
vle_fill_cols = ['total_clicks', 'num_sites']
vle_fill_cols_present = [col for col in vle_fill_cols if col in master.columns]

print(f"VLE columns to fill with 0: {vle_fill_cols_present}")

# Fill missing VLE metrics with 0 (no activity = 0 clicks)
# Use fillna without inplace to avoid Copy-on-Write warnings
for col in vle_fill_cols_present:
    before_fill = master[col].isnull().sum()
    master[col] = master[col].fillna(0)
    after_fill = master[col].isnull().sum()
    print(f"{col}: {before_fill} → {after_fill} missing values")

# Remove avg_clicks_per_day column
if 'avg_clicks_per_day' in master.columns:
    master = master.drop(columns=['avg_clicks_per_day'])
    print("\nDropped 'avg_clicks_per_day' column from master dataset")

print("\nMissing values after VLE imputation:")
print(master.isnull().sum()[master.isnull().sum() > 0])

VLE columns to fill with 0: ['total_clicks', 'num_sites']
total_clicks: 3327 → 0 missing values
num_sites: 3327 → 0 missing values

Dropped 'avg_clicks_per_day' column from master dataset

Missing values after VLE imputation:
date_unregistration    22515
dtype: int64


## Step 2: Calculating Normalized Performance (Avoiding Data Leakage)

We calculate weighted coursework scores from assessments, excluding exams to avoid data leakage. This represents student performance on graded coursework only.


### Loading Assessment Data


In [102]:
# Load student assessment scores
student_assess = pd.read_csv(PREPROCESSED_DIR / 'studentAssessment.csv')
print(f"studentAssessment: {student_assess.shape}")
print(student_assess.head())
print(f"\nColumns: {student_assess.columns.tolist()}")

# Load assessment metadata (weights)
assessments = pd.read_csv(PREPROCESSED_DIR / 'assessments.csv')
print(f"\nassessments: {assessments.shape}")
print(assessments.head())
print(f"\nColumns: {assessments.columns.tolist()}")

studentAssessment: (173912, 5)
   id_assessment  id_student  date_submitted  is_banked  score
0           1752       11391              18          0   78.0
1           1752       28400              22          0   70.0
2           1752       31604              17          0   72.0
3           1752       32885              26          0   69.0
4           1752       38053              19          0   79.0

Columns: ['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']

assessments: (206, 6)
  code_module code_presentation  id_assessment assessment_type   date  weight
0         AAA             2013J           1752             TMA   19.0    10.0
1         AAA             2013J           1753             TMA   54.0    20.0
2         AAA             2013J           1754             TMA  117.0    20.0
3         AAA             2013J           1755             TMA  166.0    20.0
4         AAA             2013J           1756             TMA  215.0    30.0

Columns: ['code_m

### Filtering Out Exams

Remove all exam records to focus on coursework (TMA/CMA) only, avoiding data leakage from future exam performance.


In [103]:
# Check assessment types
print("Assessment types in data:")
print(assessments['assessment_type'].value_counts())

# Filter out exams
assessments_no_exam = assessments[assessments['assessment_type'] != 'Exam'].copy()
# assessments_no_exam = assessments.copy()
print(f"\nAssessments after filtering exams: {assessments_no_exam.shape[0]} (removed {assessments.shape[0] - assessments_no_exam.shape[0]})")
print(f"Assessment types remaining: {assessments_no_exam['assessment_type'].unique()}")

Assessment types in data:
assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64

Assessments after filtering exams: 182 (removed 24)
Assessment types remaining: <StringArray>
['TMA', 'CMA']
Length: 2, dtype: str


### Merging Assessment Data with Weights


In [104]:
# Merge student assessments with assessment metadata to get weights
assessment_with_weights = student_assess.merge(
    assessments_no_exam[['id_assessment', 'assessment_type', 'weight', 'code_module', 'code_presentation']],
    on='id_assessment',
    how='inner'
)

print(f"Assessment records with weights: {assessment_with_weights.shape}")
print(assessment_with_weights.head())
print(f"\nColumns in assessment_with_weights: {assessment_with_weights.columns.tolist()}")

Assessment records with weights: (168953, 9)
   id_assessment  id_student  date_submitted  is_banked  score  \
0           1752       11391              18          0   78.0   
1           1752       28400              22          0   70.0   
2           1752       31604              17          0   72.0   
3           1752       32885              26          0   69.0   
4           1752       38053              19          0   79.0   

  assessment_type  weight code_module code_presentation  
0             TMA    10.0         AAA             2013J  
1             TMA    10.0         AAA             2013J  
2             TMA    10.0         AAA             2013J  
3             TMA    10.0         AAA             2013J  
4             TMA    10.0         AAA             2013J  

Columns in assessment_with_weights: ['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score', 'assessment_type', 'weight', 'code_module', 'code_presentation']


### Checking if tma_cma_weight_score was correct

The inner removes the Exam
We get weight_values < 100, because the Exam is not included

Suppose:

- TMA 40 10
- CMA 30 10
- TMA 100 0
- EXAM 70 80

The weight will be = (40 * 10 + 30*10 + 100 \* 0) / (10+10+0)

Suppose:
That student has not given one of the exams:

- TMA - 10
- CMA 30 10
- TMA 100 0
- EXAM 70 80

The weight will be = (30* 10 + 100*0) / (10+10+0)


#### Filter out exams

assessments_exam = assessments[assessments['assessment_type'] == 'Exam'].copy()

#### assessments_no_exam = assessments.copy()

print(f"\nAssessments after filtering exams: {assessments_exam.shape[0]} (removed {assessments.shape[0] - assessments_exam.shape[0]})")
print(f"Assessment types remaining: {assessments_exam['assessment_type'].unique()}")

#### Merge student assessments with assessment metadata to get weights

assessment_with_weights = student_assess.merge(
assessments_exam[['id_assessment', 'assessment_type', 'weight', 'code_module', 'code_presentation']],
on='id_assessment',
how='inner'
)

print(f"Assessment records with weights: {assessment_with_weights.shape}")
print(assessment_with_weights.head())
print(f"\nColumns in assessment_with_weights: {assessment_with_weights.columns.tolist()}")


#### Filter out exams

assessments_exam = assessments[assessments['assessment_type'] == 'Exam'].copy()

#### assessments_no_exam = assessments.copy()

print(f"\nAssessments after filtering exams: {assessments_exam.shape[0]} (removed {assessments.shape[0] - assessments_exam.shape[0]})")
print(f"Assessment types remaining: {assessments_exam['assessment_type'].unique()}")

#### Merge student assessments with assessment metadata to get weights

assessment_with_weights = student_assess.merge(
assessments_exam[['id_assessment', 'assessment_type', 'weight', 'code_module', 'code_presentation']],
on='id_assessment',
how='inner'
)

print(f"Assessment records with weights: {assessment_with_weights.shape}")
print(assessment_with_weights.head())
print(f"\nColumns in assessment_with_weights: {assessment_with_weights.columns.tolist()}")


### Calculating Weighted Scores


In [105]:
# Calculate weighted points for each assessment
assessment_with_weights['weighted_points'] = assessment_with_weights['score'] * assessment_with_weights['weight']

print("Sample weighted points calculation:")
print(assessment_with_weights[['id_student', 'score', 'weight', 'weighted_points']].head(10))

Sample weighted points calculation:
   id_student  score  weight  weighted_points
0       11391   78.0    10.0            780.0
1       28400   70.0    10.0            700.0
2       31604   72.0    10.0            720.0
3       32885   69.0    10.0            690.0
4       38053   79.0    10.0            790.0
5       45462   70.0    10.0            700.0
6       45642   72.0    10.0            720.0
7       52130   72.0    10.0            720.0
8       53025   71.0    10.0            710.0
9       57506   68.0    10.0            680.0


### Aggregating to Student Level

Group by student-module-presentation combination and sum weighted points and total available weights.


In [106]:
# CORRECTED LOGIC: Account for missing assessments
# ====================================================

# Step 1: Calculate total possible weight for EACH module-presentation
# This ensures students who miss assessments get penalized appropriately
module_total_weights = assessments_no_exam.groupby(['code_module', 'code_presentation'])['weight'].sum().reset_index()
module_total_weights.rename(columns={'weight': 'total_possible_weight'}, inplace=True)

print(f"Total possible weights per module-presentation: {module_total_weights.shape}")
print(module_total_weights.head())
print(f"\nWeight coverage statistics:")
print(module_total_weights['total_possible_weight'].describe())

# Step 2: Get all students x module-presentation combinations from master
student_module_combos = master[join_keys].drop_duplicates().copy()
print(f"\nUnique student-module-presentation combinations: {student_module_combos.shape}")

# Step 3: Merge with module total weights (baseline: all students start with 0 weighted points)
student_performance = student_module_combos.merge(
    module_total_weights,
    on=['code_module', 'code_presentation'],
    how='left'
).copy()

student_performance['sum_weighted_points'] = 0.0  # Initialize all to 0

# Step 4: Add actual submitted scores where they exist
student_submitted = assessment_with_weights.groupby(join_keys).agg({
    'weighted_points': 'sum'
}).reset_index()

student_submitted.rename(columns={'weighted_points': 'sum_weighted_points'}, inplace=True)
print(f"\nStudent submitted assessments: {student_submitted.shape}")

# Update with actual scores (merges in the submitted values)
student_performance = student_performance.drop(columns=['sum_weighted_points'])
student_performance = student_performance.merge(
    student_submitted,
    on=join_keys,
    how='left'
)
student_performance['sum_weighted_points'] = student_performance['sum_weighted_points'].fillna(0)

print(f"\nStudent performance aggregated (with missing as 0): {student_performance.shape}")
print(student_performance.head())


Total possible weights per module-presentation: (22, 3)
  code_module code_presentation  total_possible_weight
0         AAA             2013J                  100.0
1         AAA             2014J                  100.0
2         BBB             2013B                  100.0
3         BBB             2013J                  100.0
4         BBB             2014B                  100.0

Weight coverage statistics:
count     22.000000
mean      86.363636
std       35.125009
min        0.000000
25%      100.000000
50%      100.000000
75%      100.000000
max      100.000000
Name: total_possible_weight, dtype: float64

Unique student-module-presentation combinations: (32548, 3)

Student submitted assessments: (25839, 4)

Student performance aggregated (with missing as 0): (32548, 5)
   id_student code_module code_presentation  total_possible_weight  \
0       11391         AAA             2013J                  100.0   
1       28400         AAA             2013J                  100.0   
2  

### Normalization: Computing Coursework Score (0-100 scale)

Formula: `tma_cma_weighted_score = (sum_weighted_points / sum_weights_available)`

This scales the score to 0-100 based only on coursework completed, ignoring the 50% usually reserved for exams.


In [107]:
# Calculate normalized coursework score using CORRECT denominator
# Formula: (sum_weighted_points) / (total_possible_weight)
# This penalizes students who miss assessments by dividing by ALL possible weights,
# not just the weights of submitted assessments

print("="*80)
print("VERIFICATION: Weighted Score Calculation Logic")
print("="*80)

# Show sample data to verify the logic
sample = student_performance.head(10)[['id_student', 'code_module', 'code_presentation', 'sum_weighted_points', 'total_possible_weight']]
print("\nSample data:")
print(sample)

# Calculate the score
student_performance['tma_cma_weighted_score'] = (
    student_performance['sum_weighted_points'] / student_performance['total_possible_weight']
).fillna(0)

# Multiply by 100 to get 0-100 scale (since total_possible_weight < 100 in most modules)
student_performance['tma_cma_weighted_score'] = student_performance['tma_cma_weighted_score'] * 100

# Clip to [0, 100] range
student_performance['tma_cma_weighted_score'] = student_performance['tma_cma_weighted_score'].clip(0, 100)

print("\nNormalized coursework scores:")
print(student_performance[['id_student', 'code_module', 'code_presentation', 'sum_weighted_points', 'total_possible_weight', 'tma_cma_weighted_score']].head(10))
print(f"\nScore statistics:")
print(student_performance['tma_cma_weighted_score'].describe())

# Reliability check: flag modules where weight coverage is low
low_coverage = student_performance[student_performance['total_possible_weight'] < 50][['code_module', 'code_presentation', 'total_possible_weight']].drop_duplicates()
if len(low_coverage) > 0:
    print("\n⚠ WARNING: Modules with low TMA/CMA weight coverage (<50%):")
    print(low_coverage)
else:
    print("\n✓ All modules have reasonable TMA/CMA weight coverage")

VERIFICATION: Weighted Score Calculation Logic

Sample data:
   id_student code_module code_presentation  sum_weighted_points  \
0       11391         AAA             2013J               8240.0   
1       28400         AAA             2013J               6540.0   
2       30268         AAA             2013J                  0.0   
3       31604         AAA             2013J               7630.0   
4       32885         AAA             2013J               5500.0   
5       38053         AAA             2013J               6690.0   
6       45462         AAA             2013J               6780.0   
7       45642         AAA             2013J               7250.0   
8       52130         AAA             2013J               7120.0   
9       53025         AAA             2013J               7900.0   

   total_possible_weight  
0                  100.0  
1                  100.0  
2                  100.0  
3                  100.0  
4                  100.0  
5                  100.0  
6

### Merging Performance Score into Master Dataframe


In [108]:
# Master dataframe before performance merge
print(f"Master dataframe before performance merge: {master.shape}")


Master dataframe before performance merge: (32548, 17)


In [109]:
# Merge only the performance score to master dataframe
master = master.merge(
    student_performance[join_keys + ['tma_cma_weighted_score']],
    on=join_keys,
    how='left'
)

print(f"Master dataframe after performance merge: {master.shape}")
print(f"Performance score column present: {'tma_cma_weighted_score' in master.columns}")
print(f"Missing values in tma_cma_weighted_score: {master['tma_cma_weighted_score'].isnull().sum()}")


Master dataframe after performance merge: (32548, 18)
Performance score column present: True
Missing values in tma_cma_weighted_score: 0


In [110]:
# AUDIT: Verify Weighted Score Calculation Logic
# ================================================

print("="*80)
print("AUDIT: WEIGHTED SCORE CALCULATION - MISSING ASSESSMENT PENALIZATION")
print("="*80)

# Check that master has the tma_cma_weighted_score column
print(f"\nColumns in final master dataset: {master.columns.tolist()}")
print(f"tma_cma_weighted_score column exists: {'tma_cma_weighted_score' in master.columns}")

# Show score distribution
print("\nWeighted Score Distribution (after normalization):")
print(master['tma_cma_weighted_score'].describe())

# Examine submission patterns
print("\n" + "="*80)
print("FORMULA VERIFICATION")
print("="*80)
print("\nFormula being used (from aggregation):")
print("  1. total_possible_weight = sum of ALL TMA/CMA weights for module")
print("  2. sum_weighted_points = student's actual weighted score")
print("  3. tma_cma_weighted_score = (sum_weighted_points / total_possible_weight) * 100")
print("\nThis ensures that:")
print("  ✓ Missing assessments are counted as 0")
print("  ✓ Denominator = ALL possible TMA/CMA weights (not just submitted)")
print("  ✓ Students who skip assessments are appropriately penalized")

print("\nExample: If module has 3 assessments (10% each = 30% total)")
print("  - Student submits 2 with scores 50 and 80")
print("  - Score = (50*10 + 80*10 + 0*10) / 30 = 1300/30 = 43.33 points")
print("  - Final = 43.33 * 100 = ~144.4 (clipped to 100)")

print("\n⚠ Note: If total_possible_weight < 100, raw scores can exceed 100 before clipping")
print("        This is CORRECT behavior - students are normalized to their available weights")

# Show sample records
print("\n" + "="*80)
print("SAMPLE RECORDS FROM FINAL DATASET")
print("="*80)
sample_cols = ['id_student', 'code_module', 'code_presentation', 'total_clicks', 'tma_cma_weighted_score', 'final_result']
available_cols = [c for c in sample_cols if c in master.columns]
print(f"\nSample (first 10 rows):")
print(master[available_cols].head(10))


AUDIT: WEIGHTED SCORE CALCULATION - MISSING ASSESSMENT PENALIZATION

Columns in final master dataset: ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'date_registration', 'date_unregistration', 'total_clicks', 'num_sites', 'module_presentation_length', 'tma_cma_weighted_score']
tma_cma_weighted_score column exists: True

Weighted Score Distribution (after normalization):
count    32548.000000
mean        71.665049
std         44.897722
min          0.000000
25%          0.000000
50%        100.000000
75%        100.000000
max        100.000000
Name: tma_cma_weighted_score, dtype: float64

FORMULA VERIFICATION

Formula being used (from aggregation):
  1. total_possible_weight = sum of ALL TMA/CMA weights for module
  2. sum_weighted_points = student's actual weighted score
  3. tma_cma_weighted_score = (sum_weighted_points / total_possible_weight) 

In [112]:
# Merge only the performance score to master dataframe
master = master.merge(
    student_performance[join_keys + ['tma_cma_weighted_score']],
    on=join_keys,
    how='left'
)

print(f"Master dataframe after performance merge: {master.shape}")
print(f"Performance score column present: {'tma_cma_weighted_score' in master.columns}")
print(f"Missing values in tma_cma_weighted_score: {master['tma_cma_weighted_score'].isnull().sum()}")

Master dataframe after performance merge: (32548, 20)
Performance score column present: True
Missing values in tma_cma_weighted_score: 0


## Step 3: Final Feature Engineering

Create derived features for modeling: withdrawal status, registration latency, and click density.


In [113]:
print(f"Shape of dataset before feature engineering: {master.shape}")

Shape of dataset before feature engineering: (32548, 20)


### Feature 1: Withdrawal Status


In [117]:
# Create binary withdrawal flag
# 1 = student withdrawn (date_unregistration is not null)
# 0 = student completed or still registered
master['is_withdrawn'] = (~master['date_unregistration'].isnull()).astype(int)

print("Withdrawal status distribution:")
print(master['is_withdrawn'].value_counts())
print(f"Withdrawal rate: {master['is_withdrawn'].mean()*100:.2f}%")

Withdrawal status distribution:
is_withdrawn
0    22515
1    10033
Name: count, dtype: int64
Withdrawal rate: 30.83%


### Handling Remaining Missing Values


In [118]:
# 1. Get students with NaN scores
no_score_students = master[master['tma_cma_weighted_score'].isnull()]

# 2. Check how many of them withdrew early
withdrawn_early = no_score_students[no_score_students['is_withdrawn'] == 1].shape[0]

print(f"Total students with NaN scores: {len(no_score_students)}")
print(f"Of those, how many withdrew from the course: {withdrawn_early}")

Total students with NaN scores: 0
Of those, how many withdrew from the course: 0


In [119]:
print("Average clicks for students WITH scores:", master[master['tma_cma_weighted_score'].notnull()]['total_clicks'].mean())
print("Average clicks for students with NaN scores:", no_score_students['total_clicks'].mean())

Average clicks for students WITH scores: 1216.7552845028881
Average clicks for students with NaN scores: nan


In [120]:
# Fill any remaining NaNs in the performance score with 0
if 'tma_cma_weighted_score' in master.columns:
    before_fill = master['tma_cma_weighted_score'].isnull().sum()
    master['tma_cma_weighted_score'] = master['tma_cma_weighted_score'].fillna(0)
    after_fill = master['tma_cma_weighted_score'].isnull().sum()
    print(f"tma_cma_weighted_score: {before_fill} → {after_fill} missing values")

# Final check for critical columns
print("\nRemaining missing values (by column):")
remaining_missing = master.isnull().sum()
if remaining_missing.sum() > 0:
    print(remaining_missing[remaining_missing > 0])
else:
    print("No missing values in critical columns!")

tma_cma_weighted_score: 0 → 0 missing values

Remaining missing values (by column):
date_unregistration    22515
dtype: int64


## Step 4: Export & Verify

Save the final integrated master dataset and validate its structure.


### Remove is_withdrawn feature


In [121]:
master.drop(columns=['is_withdrawn'], inplace=True)

### Saving Master Dataset


In [122]:
# Save master dataframe to CSV
output_path = MERGED_DIR / 'merged.csv'
master.to_csv(output_path, index=False)

print(f"Master dataset saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / (1024**2):.2f} MB")

Master dataset saved to: ..\..\data\merged\merged.csv
File size: 3.82 MB


### Verification: Dataset Structure and Integrity


In [123]:
# Verify one row per student-module-presentation
uniqueness_check = master.groupby(join_keys).size()

print(f"Total rows in master dataset: {master.shape[0]}")
print(f"Total columns: {master.shape[1]}")
print(f"\nUnique student-module-presentation combinations: {len(uniqueness_check)}")
print(f"Max rows per combination: {uniqueness_check.max()}")
print(f"Min rows per combination: {uniqueness_check.min()}")

if uniqueness_check.max() == 1:
    print("\n✓ SUCCESS: One row per student-module-presentation!")
else:
    print(f"\n⚠ WARNING: Found {(uniqueness_check > 1).sum()} duplicate combinations")

Total rows in master dataset: 32548
Total columns: 20

Unique student-module-presentation combinations: 32548
Max rows per combination: 1
Min rows per combination: 1

✓ SUCCESS: One row per student-module-presentation!


In [124]:
# Display dataset summary
print("="*80)
print("MASTER STUDENT DATASET SUMMARY")
print("="*80)
print(f"\nShape: {master.shape[0]} rows × {master.shape[1]} columns")
print(f"\nFirst few rows:")
print(master.head())

print(f"\nColumn names and types:")
print(master.dtypes)

print(f"\nKey statistics:")
print(f"  - Modules: {master['code_module'].nunique()}")
print(f"  - Presentations: {master['code_presentation'].nunique()}")
print(f"  - Total students: {master['id_student'].nunique()}")
print(f"  - Average TMA/CMA score: {master['tma_cma_weighted_score'].mean():.2f}/100")
print(f"  - Average total clicks per student: {master['total_clicks'].mean():.2f}")

MASTER STUDENT DATASET SUMMARY

Shape: 32548 rows × 20 columns

First few rows:
  code_module code_presentation  id_student gender                region  \
0         AAA             2013J       11391      M   East Anglian Region   
1         AAA             2013J       28400      F              Scotland   
2         AAA             2013J       30268      F  North Western Region   
3         AAA             2013J       31604      F     South East Region   
4         AAA             2013J       32885      F  West Midlands Region   

       highest_education imd_band age_band  num_of_prev_attempts  \
0       HE Qualification  90-100%      55+                     0   
1       HE Qualification   20-30%    35-55                     0   
2  A Level or Equivalent   30-40%    35-55                     0   
3  A Level or Equivalent   50-60%    35-55                     0   
4     Lower Than A Level   50-60%     0-35                     0   

   studied_credits disability final_result  date_regis